# NGHIÊN CỨU VÀ ĐÁNH GIÁ PHƯƠNG PHÁP PHÁT HIỆN BACKDOOR TRÊN LoRA ADAPTER KẾT HỢP ĐẶC TRƯNG TRỌNG SỐ VÀ HÀNH VI

## Vai trò của notebook

Notebook nội bộ này chỉ kiểm tra tính toàn vẹn của gói tải lên trước khi chuyển sang notebook điều phối chính. Notebook không chuẩn bị dữ liệu, không tải mô hình, không huấn luyện và không thực hiện phép đo nghiên cứu.

## Ranh giới bằng chứng

Kết quả đạt của bước này chỉ xác nhận các tệp trong gói khớp với manifest đã phát hành. Kết quả đó không chứng minh môi trường A100, tiến độ thực nghiệm, chất lượng mô hình hoặc kết luận của đồ án.

In [ ]:
import subprocess
import sys
from pathlib import Path

sys.dont_write_bytecode = True


def has_linked_ancestor(path: Path) -> bool:
    current = path.absolute()
    while True:
        try:
            file_stat = current.stat(follow_symlinks=False)
        except OSError:
            return True
        if current.is_symlink() or getattr(file_stat, "st_file_attributes", 0) & 0x400:
            return True
        if current == current.parent:
            return False
        current = current.parent


PROJECT_ROOT = Path.cwd().absolute()
if PROJECT_ROOT.name != "lora-audit" or not (PROJECT_ROOT / "pyproject.toml").is_file():
    raise FileNotFoundError(
        "Hãy chạy notebook kiểm tra này từ thư mục lora-audit đã được đóng gói."
    )
BUNDLE_ROOT = PROJECT_ROOT.parent
BOOTSTRAP = BUNDLE_ROOT / "bootstrap.py"
VERIFIER = PROJECT_ROOT / "scripts" / "build_upload_bundle.py"
if has_linked_ancestor(BUNDLE_ROOT) or BOOTSTRAP.is_symlink() or VERIFIER.is_symlink():
    raise RuntimeError("Thư mục dự án đã đóng gói không được đi qua liên kết tượng trưng.")
if BUNDLE_ROOT.name != "TranThienNhan_TTTN" or not BOOTSTRAP.is_file() or not VERIFIER.is_file():
    raise FileNotFoundError("Thiếu thư mục gói tải lên hoặc trình kiểm tra tính toàn vẹn.")
print("Thư mục dự án:", PROJECT_ROOT)
print("Chế độ: chỉ kiểm tra tính toàn vẹn gói tải lên")

In [ ]:
subprocess.run(
    [
        sys.executable,
        str(BOOTSTRAP),
        "--mode",
        "attest",
    ],
    check=True,
)
print("Đã xác thực tính toàn vẹn gói tải lên; chưa khởi tạo môi trường, mô hình hoặc thực nghiệm.")